# 03 — Growth Correlation

**Project:** SERSA Product Demand Relationship Analysis  
**Author:** Cesar Enrique Banda Martinez  
**Date:** 2026  

---

## Context

Notebook 02 revealed 298 SKU pairs with Pearson r ≥ 0.75 on raw monthly transaction counts.  
However, raw correlation has a well-known limitation in time series analysis:

> **Two SKUs that simply grew over time will appear highly correlated even if their demand is unrelated.**

Since the main project documented a 294% revenue increase from 2022 to 2025, a shared upward trend is present across most SKUs. This trend inflates raw correlations and makes it impossible to distinguish genuine demand co-movement from coincidental parallel growth.

### Solution: correlation on monthly growth rates

Instead of asking *"do these SKUs have similar volume levels?"*, we ask:

> **"When SKU A grows faster than usual in a given month, does SKU B also grow faster than usual?"**

This is computed using `pct_change()` — the month-over-month percentage change in transaction counts — which removes the common trend and isolates true co-movement signals.

### Key finding from notebook 02 carried forward

Several top-correlated pairs (e.g. `SF1340L8 ↔ SF1340M7`, `SFPUG111T6 ↔ SFPUG111T8`) are size variants of the same base product. Their r = 1.0 in raw counts is expected — operators dispense all sizes together. This pattern persists in growth rates and is **documented as a finding, not treated as an error**.

### Handling of zero-to-nonzero transitions
`pct_change()` produces `inf` when a SKU goes from 0 transactions in one month to any positive value the next. These values are replaced with `NaN` before computing correlations. `pandas corr()` ignores `NaN` by default, so only valid month pairs contribute to each coefficient.

### Goals of this notebook
1. Load `02_pivot_filtered.csv` (86 SKUs, low-activity SKUs already removed).
2. Compute month-over-month growth rates (`pct_change`).
3. Replace `inf` / `-inf` with `NaN`.
4. Compute Pearson correlation matrix on growth rates.
5. Compare raw vs. growth-rate correlations to identify which relationships survive detrending.
6. Visualize and export results.

---

## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

---
## 2. Configuration

In [ ]:
DATA_DIR       = os.path.join(os.getcwd(), "..", "outputs", "tables")
OUTPUT_TABLES  = os.path.join(os.getcwd(), "..", "outputs", "tables")
OUTPUT_FIGURES = os.path.join(os.getcwd(), "..", "outputs", "figures")

CORR_THRESHOLD = 0.75

print(f"Data dir:       {os.path.normpath(DATA_DIR)}")
print(f"Output tables:  {os.path.normpath(OUTPUT_TABLES)}")
print(f"Output figures: {os.path.normpath(OUTPUT_FIGURES)}")

---
## 3. Load Filtered Pivot

In [ ]:
pivot = pd.read_csv(
    os.path.join(DATA_DIR, "02_pivot_filtered.csv"),
    index_col="Month",
    parse_dates=True,
    decimal=","
)

print(f"Loaded: {pivot.shape[0]} months x {pivot.shape[1]} SKUs")
print()
print(pivot.iloc[:3, :6])

---
## 4. Compute Monthly Growth Rates

`pct_change()` computes the month-over-month percentage change for each SKU.  
The first month becomes `NaN` (no previous period to compare).  
Months where a SKU transitions from 0 to any positive value produce `inf` — these are replaced with `NaN`.

In [ ]:
growth = pivot.pct_change()

# Replace inf / -inf produced by zero-to-nonzero transitions
inf_count = np.isinf(growth.values).sum()
growth.replace([np.inf, -np.inf], np.nan, inplace=True)

print(f"Growth rate matrix shape : {growth.shape}")
print(f"NaN values (first row + inf replacements) : {growth.isna().sum().sum():,}")
print(f"  of which from inf replacements           : {inf_count:,}")
print()
print(growth.iloc[1:4, :6].round(3))

---
## 5. Pearson Correlation on Growth Rates

`pandas corr()` computes pairwise correlations using only the months where both SKUs have valid (non-NaN) values.

In [ ]:
corr_growth = growth.corr(method="pearson")

print(f"Growth correlation matrix shape: {corr_growth.shape}")
print()
print(corr_growth.iloc[:4, :4].round(3))

---
## 6. Growth Rate Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(18, 16))

mask = np.triu(np.ones_like(corr_growth, dtype=bool))

sns.heatmap(
    corr_growth,
    mask=mask,
    cmap="RdYlGn",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0,
    square=True,
    cbar_kws={"shrink": 0.6, "label": "Pearson r (growth rates)"},
    ax=ax
)

ax.set_title("SKU Correlation Matrix — Monthly Growth Rates (detrended)", fontsize=16, fontweight="bold", pad=16)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", labelsize=6, rotation=90)
ax.tick_params(axis="y", labelsize=6, rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES, "03_correlation_heatmap_growth.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 03_correlation_heatmap_growth.png")

---
## 7. Top Growth-Rate Correlation Pairs

In [ ]:
corr_unstacked = (
    corr_growth
    .where(np.triu(np.ones(corr_growth.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_unstacked.columns = ["SKU_A", "SKU_B", "Pearson_r"]
corr_unstacked["Pearson_r"] = corr_unstacked["Pearson_r"].round(4)

top_positive_growth = (
    corr_unstacked[corr_unstacked["Pearson_r"] >= CORR_THRESHOLD]
    .sort_values("Pearson_r", ascending=False)
    .reset_index(drop=True)
)

top_negative_growth = (
    corr_unstacked[corr_unstacked["Pearson_r"] <= -CORR_THRESHOLD]
    .sort_values("Pearson_r", ascending=True)
    .reset_index(drop=True)
)

print(f"Pairs with r >= {CORR_THRESHOLD}  : {len(top_positive_growth)}")
print(f"Pairs with r <= -{CORR_THRESHOLD} : {len(top_negative_growth)}")
print()
print("Top 15 positive pairs (growth rates):")
print(top_positive_growth.head(15).to_string(index=False))

In [ ]:
print("Top negative pairs (growth rates):")
if len(top_negative_growth) > 0:
    print(top_negative_growth.to_string(index=False))
else:
    print(f"No pairs with r <= -{CORR_THRESHOLD} found.")

---
## 8. Raw vs. Growth Rate Comparison

A key analytical question: **which correlations survive detrending?**

Pairs that are strong in raw counts but weak in growth rates were likely driven by shared trend.  
Pairs that remain strong in both are the most analytically meaningful.

In [ ]:
# Load raw correlation matrix from notebook 02
corr_raw = pd.read_csv(
    os.path.join(DATA_DIR, "02_correlation_matrix.csv"),
    index_col=0,
    decimal=","
)

# Extract upper triangle for both matrices — aligned to same SKU set
shared_skus = corr_raw.index.intersection(corr_growth.index)
corr_raw_shared    = corr_raw.loc[shared_skus, shared_skus]
corr_growth_shared = corr_growth.loc[shared_skus, shared_skus]

mask_upper = np.triu(np.ones(corr_raw_shared.shape, dtype=bool), k=1)

raw_vals    = corr_raw_shared.values[mask_upper]
growth_vals = corr_growth_shared.values[mask_upper]

# Remove pairs where growth corr is NaN (insufficient valid months)
valid = ~np.isnan(growth_vals)
raw_vals    = raw_vals[valid]
growth_vals = growth_vals[valid]

print(f"Valid pairs for comparison: {valid.sum():,}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

ax.scatter(raw_vals, growth_vals, alpha=0.25, s=8, color="#2C7BB6")
ax.axhline(y=CORR_THRESHOLD,  color="#D7191C", linestyle="--", linewidth=0.8, alpha=0.7)
ax.axvline(x=CORR_THRESHOLD,  color="#D7191C", linestyle="--", linewidth=0.8, alpha=0.7)
ax.axhline(y=-CORR_THRESHOLD, color="#D7191C", linestyle="--", linewidth=0.8, alpha=0.7)
ax.axvline(x=-CORR_THRESHOLD, color="#D7191C", linestyle="--", linewidth=0.8, alpha=0.7)
ax.axline((0, 0), slope=1, color="gray", linestyle="-", linewidth=0.6, alpha=0.5)

ax.set_xlabel("Raw correlation (notebook 02)", fontsize=11)
ax.set_ylabel("Growth-rate correlation (notebook 03)", fontsize=11)
ax.set_title("Raw vs. Detrended Correlation — All SKU Pairs", fontsize=13, fontweight="bold")
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.grid(alpha=0.2)
sns.despine()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES, "03_raw_vs_growth_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 03_raw_vs_growth_scatter.png")

### Interpreting the scatter plot

- **Top-right quadrant** (raw high, growth high): genuine demand co-movement — strong in both layers.
- **Bottom-right quadrant** (raw high, growth low): spurious correlation driven by shared trend only.
- **Diagonal line**: perfect agreement between raw and growth correlations.

Pairs far below the diagonal were inflated by the common growth trend and are less reliable as demand signals.

---
## 9. Top Growth Pairs Bar Chart

In [ ]:
top_n = min(20, len(top_positive_growth))

if top_n > 0:
    plot_data = top_positive_growth.head(top_n).copy()
    plot_data["pair"] = plot_data["SKU_A"] + "  ⇔  " + plot_data["SKU_B"]

    fig, ax = plt.subplots(figsize=(10, top_n * 0.45 + 1))

    ax.barh(
        plot_data["pair"][::-1],
        plot_data["Pearson_r"][::-1],
        color="#2C7BB6",
        alpha=0.85
    )

    ax.axvline(x=CORR_THRESHOLD, color="#D7191C", linestyle="--", linewidth=1, label=f"Threshold r={CORR_THRESHOLD}")
    ax.set_xlim(0, 1)
    ax.set_xlabel("Pearson r (growth rates)", fontsize=11)
    ax.set_title(f"Top {top_n} Strongest Growth-Rate Correlations", fontsize=13, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(axis="x", alpha=0.3)
    sns.despine()

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_FIGURES, "03_top_growth_pairs.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: 03_top_growth_pairs.png")
else:
    print(f"No pairs above threshold {CORR_THRESHOLD} to plot.")

---
## 10. Export

In [ ]:
corr_growth.to_csv(
    os.path.join(OUTPUT_TABLES, "03_correlation_matrix_growth.csv"),
    decimal=","
)

top_positive_growth.to_csv(
    os.path.join(OUTPUT_TABLES, "03_top_positive_pairs_growth.csv"),
    index=False,
    decimal=","
)

top_negative_growth.to_csv(
    os.path.join(OUTPUT_TABLES, "03_top_negative_pairs_growth.csv"),
    index=False,
    decimal=","
)

print("Export complete.")
print(f"  03_correlation_matrix_growth.csv     -> {corr_growth.shape}")
print(f"  03_top_positive_pairs_growth.csv     -> {len(top_positive_growth)} pairs")
print(f"  03_top_negative_pairs_growth.csv     -> {len(top_negative_growth)} pairs")

---
## 11. Summary

| Output | Description |
|--------|-------------|
| `03_correlation_matrix_growth.csv` | 86 × 86 Pearson correlation matrix on growth rates |
| `03_top_positive_pairs_growth.csv` | Pairs with growth-rate r ≥ 0.75 |
| `03_top_negative_pairs_growth.csv` | Pairs with growth-rate r ≤ −0.75 |
| `03_correlation_heatmap_growth.png` | Heatmap — detrended correlations |
| `03_raw_vs_growth_scatter.png` | Raw vs. growth-rate correlation scatter |
| `03_top_growth_pairs.png` | Top 20 growth-rate pairs bar chart |

### Key questions this notebook answers
- How many of the 298 raw pairs survive detrending?
- Which SKU relationships are genuine demand co-movement vs. shared trend?
- Do the size-variant clusters (SF1340 family, SFPUG111 family) remain correlated after detrending?

**Next:** `04_lead_lag_analysis.ipynb` — cross-correlation at lags 1–6 months to identify whether any SKU anticipates the behavior of another.